# Colab setup and safe ConceptNet input

Default mode is an offline tiny fixture. Set `ALLOW_PRODUCTION_DOWNLOAD` only after reviewing the resource preflight and source/checksum status.

In [ ]:
from pathlib import Path
from semmap_haken.config import load_config
from semmap_haken.data_manager import DatasetDescriptor, acquire_dataset
from semmap_haken.notebook import (detect_environment, make_execution_metadata, mount_drive, resolve_notebook_paths, run_resource_preflight)

ALLOW_PRODUCTION_DOWNLOAD = False
USE_DRIVE = False
CONFIG_PATH = Path('configs/conceptnet_en_smoke.yaml')
MANUAL_DATASET_PATH = None  # e.g. Path('/content/uploaded.csv.gz'); never a personal Drive path
config = load_config(CONFIG_PATH)
paths = resolve_notebook_paths(CONFIG_PATH)
environment = detect_environment()
# The tiny default fixture has no production resource requirement; production remains preflight-gated.
preflight = run_resource_preflight(config.runtime.resource_profile if ALLOW_PRODUCTION_DOWNLOAD else None, paths.data_root)
print({'environment': environment.kind, 'preflight_ok': preflight.ok, 'workspace': str(paths.workspace_root), 'cache': str(paths.cache_root)})

In [ ]:
# Colab-only adapter boundary. The core package never imports google.colab.
def colab_drive_mount():
    from google.colab import drive
    return drive.mount('/content/drive')

mount_drive(enabled=USE_DRIVE, mount_callback=colab_drive_mount if USE_DRIVE else None)
fixture = Path('tests/fixtures/conceptnet_tiny.tsv')
if MANUAL_DATASET_PATH is not None:
    result = acquire_dataset(DatasetDescriptor(dataset_id=config.dataset.source), cache_root=paths.cache_root, manual_path=MANUAL_DATASET_PATH)
elif ALLOW_PRODUCTION_DOWNLOAD:
    result = acquire_dataset(DatasetDescriptor(dataset_id=config.dataset.source), cache_root=paths.cache_root)
else:
    result = acquire_dataset(DatasetDescriptor(dataset_id='offline-tiny', filename=fixture.name), cache_root=paths.cache_root, manual_path=fixture)
print({'source': result.source, 'path': str(result.path), 'sha256': result.sha256, 'cache_hit': result.cache_hit})

In [ ]:
metadata = make_execution_metadata(notebook_name='00_colab_setup_and_conceptnet.ipynb', environment=environment, paths=paths, drive_enabled=USE_DRIVE)
metadata